# Philippine SMS Scam Detector: EDA

Load the base CSV from `data/raw/spam.csv` (Kaggle `v1`/`v2` or `label`/`message` columns) and the optional Philippine supplement. This notebook gives a small, reproducible overview before training.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.train import load_dataset, load_training_data
from src.preprocess import extract_features

pd.set_option('display.max_colwidth', 80)

In [ ]:
messages, labels = load_training_data()
df = pd.DataFrame({'label': labels, 'message': messages})
df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
df['message_length'] = df['message'].str.len()
df.groupby('label')['message_length'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], title='Message counts by label', ylabel='Messages')
axes[1].hist(df.loc[df.label == 'ham', 'message_length'], bins=40, alpha=0.6, label='ham')
axes[1].hist(df.loc[df.label == 'spam', 'message_length'], bins=40, alpha=0.6, label='spam')
axes[1].set_title('Message length by label'); axes[1].set_xlabel('Characters'); axes[1].legend()
plt.tight_layout()

In [ ]:
# Indicator features on representative Philippine traffic
examples = [
    'Nag-claim ka ng ₱500,000 sa GMA Kapuso Foundation. Call now 09123456789',
    'BDO: Your account is locked. Verify here: http://bit.ly/fakebdo',
    'Congrats! You won ₱1M from PCSO. Text your name and address to 0922xxxxxxx',
    'Your BDO OTP is 583920. Do not share this code with anyone.',
    'GCash: You received ₱1,500 from ANA LIM. Balance: ₱6,200.00.',
    'Uwi ka na agad, ulan na dito sa amin. Ingat!',
]
pd.DataFrame([extract_features(m) | {'message': m} for m in examples])

## Philippine data checklist

The base dataset is English-heavy. The synthetic supplement (`data/raw/ph_supplement.csv`) adds template-based scam and legitimate examples so the pipeline can be exercised locally — replace or extend it with consented, de-identified real examples and keep a separate holdout (`data/raw/ph_holdout.csv`). Legitimate OTPs, bank alerts, receipts and delivery notices must stay labeled `ham` so evaluation can measure harmful false positives.